---
course_code: 02BMSBI24365
course_title: AI and ML in Bioinformatics
unit: Unit 2 — Machine Learning Techniques
submodule: 2.4 Introduction to ML Pipelines
document_type: Practical Laboratory Notebook
ai_tier: Full AI (Learning Aid)
---

# 🧬 Master End-to-End Biological ML Pipeline

<small><b>Course Code:</b> 02BMSBI24365 | <b>Unit 2:</b> Machine Learning Techniques (Submodule 2.4) | <b>Environment:</b> <code>gcu-aiml-bioinfo</code> | <b>AI Tier:</b> Full AI (Learning Aid)</small>

---

## The Universal 4-Step Machine Learning Architecture

Every machine learning project in computational biology follows an identical 4-step sequence:
$$\text{1. Dataset Preparation} \longrightarrow \text{2. Model Selection} \longrightarrow \text{3. Training} \longrightarrow \text{4. Evaluation}$$

This notebook implements this architecture across two clinical benchmarks side-by-side using Scikit-learn `Pipeline` objects to enforce the **Zero Data Leakage Invariant**:
* **Track A (Classification):** Breast Cancer Wisconsin dataset $\to$ Malignant vs. Benign diagnosis.
* **Track B (Regression):** Diabetes clinical cohort $\to$ Quantitative disease progression prediction.

Each step contains separate collapsible sections for Track A and Track B.

## Step 0: Reproducible Environment & Pipeline Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay, classification_report, roc_curve, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Helvetica, Arial, DejaVu Sans'
plt.rcParams['figure.dpi'] = 110
RANDOM_STATE = 42

print("✓ Environment initialized. Scikit-learn Pipeline ready.")

---
## Step 1: Dataset Preparation (Data Preprocessing)

In this foundational step, we inspect features, verify biological labels, and partition data into train and test splits **before** any transformations are computed.

<details>
<summary><b>▶ Click to Expand: Track A — Breast Cancer Wisconsin (Classification) Data Preparation</b></summary>

### Track A: Diagnostic Cancer Dataset
* **Cohort:** 569 patients, 30 cellular morphometric features.
* **Target:** Binary diagnosis (`1 = Malignant`, `0 = Benign`).
* **Strategy:** Use `stratify=y` during splitting to preserve the natural cancer prevalence across training and test subsets.
</details>

In [ ]:
# Track A: Prepare Breast Cancer Dataset
cancer_bunch = load_breast_cancer(as_frame=True)
X_cancer = cancer_bunch.data
# Re-map so 1 = Malignant, 0 = Benign
y_cancer = (cancer_bunch.target == 0).astype(int)

# Stratified partition: 80% train, 20% test
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cancer, y_cancer, test_size=0.20, random_state=RANDOM_STATE, stratify=y_cancer
)

print(f"[Track A] Training samples: {X_train_c.shape[0]} | Test samples: {X_test_c.shape[0]}")
print(f"[Track A] Training Malignant prevalence: {y_train_c.mean()*100:.1f}%")
print(f"[Track A] Test Malignant prevalence:     {y_test_c.mean()*100:.1f}%")

<details>
<summary><b>▶ Click to Expand: Track B — Diabetes Cohort (Regression) Data Preparation</b></summary>

### Track B: Diabetes Quantitative Progression
* **Cohort:** 442 patients, 10 physiological variables.
* **Target:** Quantitative disease progression index recorded one year after baseline.
* **Strategy:** Standard continuous random train/test split.
</details>

In [ ]:
# Track B: Prepare Diabetes Dataset
diabetes_bunch = load_diabetes(as_frame=True)
X_diab = diabetes_bunch.data
y_diab = diabetes_bunch.target

# Continuous partition: 80% train, 20% test
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_diab, y_diab, test_size=0.20, random_state=RANDOM_STATE
)

print(f"[Track B] Training samples: {X_train_r.shape[0]} | Test samples: {X_test_r.shape[0]}")
print(f"[Track B] Progression target range: [{y_diab.min():.1f}, {y_diab.max():.1f}]")

---
## Step 2: Model Selection & Leak-Free Pipeline Assembly

We bundle transformers and estimators into a single `Pipeline`. This prevents **data leakage** by ensuring scaling parameters (means, medians, variances) are computed strictly on the training partition during `.fit()`.

<details>
<summary><b>▶ Click to Expand: Track A — Classification Pipeline Architecture</b></summary>

### Pipeline Architecture:
$$\text{Raw Nuclear Morphometry Features} \longrightarrow [\text{StandardScaler}] \longrightarrow [\text{LogisticRegression}]$$
* `StandardScaler`: Centers features to zero-mean and unit-variance.
* `LogisticRegression`: Calculates posterior probability $P(\text{Malignant} \mid X)$.
</details>

In [ ]:
# Track A: Build Classifier Pipeline
pipeline_clf = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

print("Track A Pipeline Structure:")
print(pipeline_clf)

<details>
<summary><b>▶ Click to Expand: Track B — Regression Pipeline Architecture</b></summary>

### Pipeline Architecture:
$$\text{Raw Physiological Features} \longrightarrow [\text{RobustScaler}] \longrightarrow [\text{Ridge Regression}]$$
* `RobustScaler`: Centers features using median and IQR, resisting extreme outlier patient values.
* `Ridge`: Linear regression with $L_2$ regularized penalty to stabilize correlated biological measurements.
</details>

In [ ]:
# Track B: Build Regressor Pipeline
pipeline_reg = Pipeline([
    ('scaler', RobustScaler()),
    ('regressor', Ridge(alpha=1.0, random_state=RANDOM_STATE))
])

print("Track B Pipeline Structure:")
print(pipeline_reg)

---
## Step 3: Training

In Step 3, the entire pipeline is fitted with a single `.fit()` call. The pipeline automatically fits the scaler on the training features, transforms them, and passes the scaled values directly into the estimator.

<details>
<summary><b>▶ Click to Expand: Track A — Training the Diagnostic Classifier</b></summary>

We fit `pipeline_clf` on the training partition:
$$\text{pipeline\_clf.fit}(X_{\text{train}}, y_{\text{train}})$$
</details>

In [ ]:
# Track A: Train Classification Pipeline
pipeline_clf.fit(X_train_c, y_train_c)
print("✓ Track A Diagnostic Classifier trained successfully.")

<details>
<summary><b>▶ Click to Expand: Track B — Training the Quantitative Regressor</b></summary>

We fit `pipeline_reg` on the continuous training partition:
$$\text{pipeline\_reg.fit}(X_{\text{train}}, y_{\text{train}})$$
</details>

In [ ]:
# Track B: Train Regression Pipeline
pipeline_reg.fit(X_train_r, y_train_r)
print("✓ Track B Quantitative Regressor trained successfully.")

---
## Step 4: Evaluation

We evaluate model performance on the unseen test set. The pipeline applies the stored training transformation parameters to test features before generating predictions, preventing data leakage.

<details>
<summary><b>▶ Click to Expand: Track A — Classification Evaluation (Confusion Matrix & ROC-AUC)</b></summary>

We evaluate the diagnostic classifier using:
1. **Confusion Matrix**
2. **Precision, Recall (Sensitivity), Specificity, and F1-Score**
3. **ROC-AUC Diagnostic Discrimination**
</details>

In [ ]:
# Track A: Evaluate Classification Pipeline
y_pred_c = pipeline_clf.predict(X_test_c)
y_prob_c = pipeline_clf.predict_proba(X_test_c)[:, 1]

cm_c = confusion_matrix(y_test_c, y_pred_c)
tn_c, fp_c, fn_c, tp_c = cm_c.ravel()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Subplot 1: Confusion Matrix
ConfusionMatrixDisplay(confusion_matrix=cm_c, display_labels=['Benign', 'Malignant']).plot(
    cmap='Blues', ax=axes[0], colorbar=False
)
axes[0].set_title("Track A: Confusion Matrix", fontsize=11, fontweight='bold')
axes[0].grid(False)

# Subplot 2: ROC Curve
fpr_c, tpr_c, _ = roc_curve(y_test_c, y_prob_c)
auc_c = roc_auc_score(y_test_c, y_prob_c)
axes[1].plot(fpr_c, tpr_c, color='#1b6ca8', lw=2.2, label=f'Pipeline (AUC = {auc_c:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1.2)
axes[1].set_xlabel('False Positive Rate (1 - Specificity)')
axes[1].set_ylabel('True Positive Rate (Sensitivity)')
axes[1].set_title(f'Track A: ROC Curve (AUC = {auc_c:.3f})', fontsize=11, fontweight='bold')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()

print("=== Track A: Detailed Classification Performance ===")
print(classification_report(y_test_c, y_pred_c, target_names=['Benign', 'Malignant'], digits=4))

<details>
<summary><b>▶ Click to Expand: Track B — Regression Evaluation (MAE, RMSE, R² & Residuals)</b></summary>

We evaluate the continuous prediction model using:
1. **Mean Absolute Error (MAE)**
2. **Root Mean Squared Error (RMSE)**
3. **Coefficient of Determination ($R^2$)**
4. **Observed vs. Predicted Residual Diagnostics**
</details>

In [ ]:
# Track B: Evaluate Regression Pipeline
y_pred_r = pipeline_reg.predict(X_test_r)

mae_r = mean_absolute_error(y_test_r, y_pred_r)
rmse_r = np.sqrt(mean_squared_error(y_test_r, y_pred_r))
r2_r = r2_score(y_test_r, y_pred_r)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Subplot 1: Observed vs Predicted
axes[0].scatter(y_test_r, y_pred_r, color='#2ca02c', alpha=0.75, edgecolor='black', s=45)
axes[0].plot([y_test_r.min(), y_test_r.max()], [y_test_r.min(), y_test_r.max()], 'r--', lw=1.5)
axes[0].set_xlabel('True Observed Disease Progression')
axes[0].set_ylabel('Predicted Disease Progression')
axes[0].set_title(f'Track B: Observed vs. Predicted (R² = {r2_r:.3f})', fontsize=11, fontweight='bold')

# Subplot 2: Residuals Distribution
residuals_r = y_test_r - y_pred_r
sns.histplot(residuals_r, kde=True, ax=axes[1], color='#1f77b4', edgecolor='black', bins=15)
axes[1].axvline(0, color='red', linestyle='--', lw=1.5)
axes[1].set_xlabel('Prediction Error (Residual)')
axes[1].set_title(f'Track B: Residuals (MAE = {mae_r:.1f}, RMSE = {rmse_r:.1f})', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"Track B Performance: MAE = {mae_r:.2f} | RMSE = {rmse_r:.2f} | R² = {r2_r:.4f}")

---
## Verification: Leak-Free Cross-Validation

The ultimate advantage of Scikit-learn `Pipeline` encapsulation is that cross-validation (`cross_val_score`) fits preprocessing transformations strictly on the training folds inside each cross-validation split, completely eliminating data leakage.

In [ ]:
# 5-Fold Stratified Cross-Validation on Track A
cv_c = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores_c = cross_val_score(pipeline_clf, X_cancer, y_cancer, cv=cv_c, scoring='roc_auc')

# 5-Fold Cross-Validation on Track B
cv_r = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores_r = cross_val_score(pipeline_reg, X_diab, y_diab, cv=cv_r, scoring='r2')

print(f"Track A 5-Fold ROC-AUC: {cv_scores_c.mean():.3f} (+/- {cv_scores_c.std():.3f})")
print(f"Track B 5-Fold R²:      {cv_scores_r.mean():.3f} (+/- {cv_scores_r.std():.3f})")